# Product

Adapted from Qualtran v0.7.0, commit bcb17bb25360ed68ce3bc084645db3e929799389.
Copyright Google LLC. SPDX-License-Identifier: Apache-2.0.

In [ ]:
from math import pi
from collections import Counter

import torch
from IPython.display import Markdown, SVG, display

from examples.algorithms.field_addition_catalog import field_signature_svg
from examples.algorithms.product_block_catalog import (
    product_child, product_child_binder, product_example_signature, product_symbolic_example,
)
from flagquantum import Circuit
from flagquantum.compiler._component_trace.call_graph import modeled_call_graph
from flagquantum.compiler._component_trace.resource_model import ComponentCost, GateCost
from flagquantum.drawer.call_graph import (
    export_modeled_call_graph_dot, format_call_graph_markdown, render_call_graph_svg,
)
from flagquantum.simulation.unitary import get_unitary

## Product of block encodings

To encode $U_1 U_2 \cdots U_n$ on a common system, execute the children
from right to left. For child normalizations $\alpha_i$, ancilla widths
$a_i$ and errors $\epsilon_i$, the product declares
$\prod_i\alpha_i$, $n-1+\max_i a_i$ and $\sum_i \alpha_i\epsilon_i$.

The first $n-1$ ancilla bits are success flags. After the relevant child
runs, a zero check of its shared ancilla controls its flag. The remaining
ancilla bits form the shared workspace for the child signal registers.
The packed resource register uses the largest child resource width and
requires the source zero-restoration promise. Private temporary work is
reserved separately from these declared ports.

The symbolic example describes the port widths and scalar metadata.
Physical binding uses concrete widths.

[Quantum algorithms: A survey of applications and end-to-end complexities](https://arxiv.org/abs/2310.03011)
describes the product construction.

In [ ]:
from flagquantum.algorithms.primitives.product_block_encoding import ProductBlockEncoding

### Example Instances

In [ ]:
t_child, h_child = product_child("t"), product_child("h")
product = ProductBlockEncoding((t_child, h_child))

In [ ]:
u1 = product_child("t", alpha=0.5, ancilla=2, resource=1, epsilon=0.01)
u2 = product_child("h", alpha=0.5, ancilla=1, resource=1, epsilon=0.1)
properties = ProductBlockEncoding((u1, u2))
assert (properties.alpha, properties.ancilla_bitsize, properties.resource_bitsize) == (0.25, 3, 1)

In [ ]:
symbolic_signature, symbolic_alpha, symbolic_error = product_symbolic_example()
print("Symbolic normalization", symbolic_alpha)
print("Symbolic error", symbolic_error)

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([
    ("Product", product_example_signature(product), {}),
    ("Product with auxiliaries", product_example_signature(properties), {}),
    ("Product with symbolic ancilla width", symbolic_signature, {}),
])))

### Call Graph

In [ ]:
cost = product.resource_model((
    ComponentCost("T block", 1, GateCost(t=1)),
    ComponentCost("Hadamard block", 1, GateCost(clifford=1)),
))
graph = modeled_call_graph(cost, max_depth=1)
display(SVG(render_call_graph_svg(export_modeled_call_graph_dot(cost, max_depth=1))))
display(Markdown(format_call_graph_markdown(
    graph.root, graph.direct_calls, graph.leaf_counts, title=lambda node: node.name,
)))

### Verify the product through append

The source example composes T and Hadamard on the same system, with an
idle success flag. Check the complete complex matrix and append the inverse.
The matrix comparison fixes the execution order because T and Hadamard
do not commute.

In [ ]:
binders = (product_child_binder(t_child), product_child_binder(h_child))
binding = dict(system=(0,), ancilla=(1,), workspaces=((), ()), classical_bits=((), ()))
circuit = Circuit(2, dtype=torch.complex128)
circuit.append(product.make_on(binders, **binding))
h = torch.tensor([[1, 1], [1, -1]], dtype=torch.complex128) / 2**0.5
t = torch.diag(torch.exp(torch.tensor([0j, 0.25j * pi], dtype=torch.complex128)))
identity = torch.eye(2, dtype=torch.complex128)
torch.testing.assert_close(get_unitary(circuit), torch.kron(t @ h, identity),
                           atol=2e-12, rtol=0)
print("Emitted primitives", dict(Counter(gate.name for gate in circuit.to_ir().instructions)))
circuit.append(product.make_on(binders, **binding, adjoint=True))
torch.testing.assert_close(get_unitary(circuit), torch.eye(4, dtype=torch.complex128),
                           atol=2e-12, rtol=0)